# Datacube Contracts — custom hooks

Demonstrates `DatacubeContract` with custom validators, transformers, and loaders.

| # | Topic |
|---|---|
| 1 | DatacubeContract — loader-only contract |
| 2 | Adding a request validator |
| 3 | Adding a request transformer |
| 4 | Adding a frame transformer |
| 5 | Full contract with all hooks |
| 6 | DatacubeResource integration |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "boti").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import pandas as pd

from boti_data import (
    DatacubeConfig,
    DatacubeContract,
    DatacubeResource,
)

## 1. DatacubeContract — loader-only

The simplest contract just provides a loader function.

In [2]:
def simple_loader(cube_name: str) -> pd.DataFrame:
    """Return a DataFrame based on the cube name."""
    if cube_name == "users":
        return pd.DataFrame({"id": [1, 2], "name": ["alice", "bob"]})
    elif cube_name == "orders":
        return pd.DataFrame({"order_id": [101, 102], "total": [50.0, 75.0]})
    raise ValueError(f"Unknown cube: {cube_name}")

contract = DatacubeContract(loader=simple_loader)
print(f"Contract: loader={contract.loader is not None}, validator={contract.request_validator is not None}")

Contract: loader=True, validator=False


## 2. Adding a request validator

Validators can reject invalid requests before the loader runs.

In [3]:
def validate_request(request: dict) -> None:
    """Ensure the request has required fields."""
    if not isinstance(request, dict):
        raise TypeError("Request must be a dict")
    if "cube" not in request:
        raise ValueError("Request must include 'cube'")
    if request.get("limit", 0) < 0:
        raise ValueError("limit must be non-negative")

contract_v = DatacubeContract(
    loader=simple_loader,
    request_validator=validate_request,
)

# Test the validator
try:
    contract_v.request_validator({"cube": "users"})
    print("Valid request: passed")
except ValueError as e:
    print(f"Validation error: {e}")

try:
    contract_v.request_validator({"limit": -1})
except ValueError as e:
    print(f"Rejected bad request: {e}")

Valid request: passed
Rejected bad request: Request must include 'cube'


## 3. Adding a request transformer

Transformers modify the request before passing it to the loader.

In [4]:
def transform_request(request: dict) -> dict:
    """Add defaults and normalize the request."""
    transformed = dict(request)
    transformed.setdefault("cube", "users")
    transformed.setdefault("limit", 100)
    transformed["offset"] = transformed.get("offset", 0)
    return transformed

contract_t = DatacubeContract(
    loader=lambda req: simple_loader(req["cube"]),
    request_transformer=transform_request,
)

result = contract_t.request_transformer({})
print(f"Transformed request: {result}")

Transformed request: {'cube': 'users', 'limit': 100, 'offset': 0}


## 4. Adding a frame transformer

Frame transformers post-process the loaded DataFrame (e.g., add columns, filter rows).

In [5]:
def add_metadata(frame: pd.DataFrame, request: dict) -> pd.DataFrame:
    """Annotate the frame with metadata from the request."""
    df = frame.copy()
    df["_source"] = request.get("cube", "unknown")
    df["_loaded_at"] = pd.Timestamp.now()
    return df

contract_f = DatacubeContract(
    loader=simple_loader,
    frame_transformer=add_metadata,
)

frame = contract_f.loader("users")
augmented = contract_f.frame_transformer(frame, {"cube": "users"})
print("Frame with metadata:")
print(augmented)

Frame with metadata:
   id   name _source                 _loaded_at
0   1  alice   users 2026-06-22 19:09:29.946541
1   2    bob   users 2026-06-22 19:09:29.946541


## 5. Full contract with all hooks

Hooks execute in this order:
1. request_transformer → 2. request_validator → 3. loader → 4. frame_transformer

In [6]:
def full_loader(req: dict) -> pd.DataFrame:
    """Use the normalized request to load data."""
    return pd.DataFrame({
        "item": [f"{req['cube']}_{i}" for i in range(req["limit"])],
        "value": [float(i) for i in range(req["limit"])],
    })

full_contract = DatacubeContract(
    request_transformer=transform_request,
    request_validator=validate_request,
    loader=full_loader,
    frame_transformer=add_metadata,
)

# Simulate the full pipeline
raw_request = {"limit": 5}
transformed = full_contract.request_transformer(raw_request)
full_contract.request_validator(transformed)
frame = full_contract.loader(transformed)
result = full_contract.frame_transformer(frame, transformed)
print("Full pipeline result:")
print(result)

Full pipeline result:
      item  value _source                 _loaded_at
0  users_0    0.0   users 2026-06-22 19:09:29.959113
1  users_1    1.0   users 2026-06-22 19:09:29.959113
2  users_2    2.0   users 2026-06-22 19:09:29.959113
3  users_3    3.0   users 2026-06-22 19:09:29.959113
4  users_4    4.0   users 2026-06-22 19:09:29.959113


## 6. DatacubeResource integration

`DatacubeResource` wraps a `DatacubeConfig` (which itself wraps a `DatacubeContract`) and integrates with the `ManagedResource` lifecycle.

In [7]:
config = DatacubeConfig(
    contract=DatacubeContract(
        loader=simple_loader,
    ),
    default_cube="users",
    verbose=True,
)

resource = DatacubeResource(config=config)
print(f"Resource: {resource}")
print(f"  default_cube: {resource.config.default_cube}")
print(f"  has_contract: {resource.config.contract is not None}")

# The contract's loader is accessible via the config
if resource.config.contract and resource.config.contract.loader:
    data = resource.config.contract.loader("orders")
    print(f"\nLoaded 'orders' cube:\n{data}")

resource.close()

Resource: <boti_data.datacube.resource.DatacubeResource object at 0x112897620>
  default_cube: users
  has_contract: True

Loaded 'orders' cube:
   order_id  total
0       101   50.0
1       102   75.0


### Summary

- **`DatacubeContract`** — typed hooks for request transformation, validation, loading, and frame post-processing.
- Hooks execute in the order: transformer → validator → loader → frame_transformer.
- **`DatacubeConfig`** wraps a contract with `ManagedResource` lifecycle integration.
- **`DatacubeResource`** provides the full lifecycle-managed runtime for contract-based data loading.